# ESR 2024 regional and country sensitivity comparison

Calculates percentage changes in **Total Cost** versus the 2024 base case for all countries and the selected European regions. Regional costs are weighted by fixed methane demand from the base input file.

In [ ]:
import os
import pandas as pd


# ============================================================
# 1. SETTINGS
# ============================================================

base_scenario = "run_2024"

sensitivity_types = [
    "lng",
    "pipe",
    "prod",
    "regas",
    "liqu",
    "channel",
]

sensitivity_levels = [
    "10pup",
    "10pdown",
    "05pup",
    "05pdown",
]


# ============================================================
# 2. PATHS
# ============================================================

# Input data folder. Both the base input file and the
# country-region file are stored here.
input_file_path = os.path.join(
    '..',
    '..',
    '01_data',
    '01_input_data',
    '02_processed',
    '02_paper_ESR'
)

full_input_data_path = os.path.abspath(
    os.path.join(
        os.getcwd(),
        input_file_path
    )
)

# Prepared results folder
prepared_results_path = os.path.join(
    '..',
    '..',
    '01_data',
    '02_output_data',
    '02_unidirectional_results',
    '02_paper_ESR',
    '02_prepared_results'
)

full_prepared_results_path = os.path.abspath(
    os.path.join(
        os.getcwd(),
        prepared_results_path
    )
)


# ============================================================
# 3. INPUT FILES
# ============================================================

# Base input file containing methane demand in the Supply sheet
input_file_name = f"inputs_ESR_2026_{base_scenario}.xlsx"

full_input_file_path = os.path.join(
    full_input_data_path,
    input_file_name
)

# Country/region definition file
region_file_name = "country_regions_ESR_2024.xlsx"

full_region_file_path = os.path.join(
    full_input_data_path,
    region_file_name
)


# ============================================================
# 4. PREPARED RESULT FILE PATH
# ============================================================

def get_prepared_results_file(scenario_name):
    """Construct the full path to a prepared costs/shares result file."""
    file_name = f"costs_shares_ESR_2026_{scenario_name}.xlsx"
    return os.path.join(full_prepared_results_path, file_name)


base_file = get_prepared_results_file(base_scenario)


# ============================================================
# 5. CREATE ALL 24 SENSITIVITY FILE PATHS
# ============================================================

sensitivity_files = {}

for sensitivity_type in sensitivity_types:
    for sensitivity_level in sensitivity_levels:
        scenario_name = (
            f"{base_scenario}_sens_"
            f"{sensitivity_type}_cost_"
            f"{sensitivity_level}"
        )

        sensitivity_name = f"{sensitivity_type}_{sensitivity_level}"

        sensitivity_files[sensitivity_name] = get_prepared_results_file(
            scenario_name
        )


# ============================================================
# 6. CHECK REQUIRED FILES
# ============================================================

if not os.path.exists(base_file):
    raise FileNotFoundError(
        f"Base result file not found:\n{base_file}"
    )

if not os.path.exists(full_input_file_path):
    raise FileNotFoundError(
        f"Input file not found:\n{full_input_file_path}"
    )

if not os.path.exists(full_region_file_path):
    raise FileNotFoundError(
        f"Region file not found:\n{full_region_file_path}"
    )

missing_files = [
    name
    for name, file_path in sensitivity_files.items()
    if not os.path.exists(file_path)
]

if missing_files:
    raise FileNotFoundError(
        "The following sensitivity files are missing:\n"
        + "\n".join(missing_files)
    )


# ============================================================
# 7. LOAD COUNTRY GROUPS FROM EXCEL
# ============================================================

def load_country_groups(region_file):
    """
    Read the country-region definitions.

    Only the four required region columns are used. Any other
    columns in the workbook, such as 'Primary Subregion', are ignored.
    """
    df_regions = pd.read_excel(
        region_file,
        sheet_name="regions"
    )

    df_regions["Node"] = (
        df_regions["Node"]
        .astype(str)
        .str.strip()
    )

    region_columns = [
        "Europe",
        "North-Eastern Europe",
        "Central-Eastern Europe",
        "South-Eastern Europe",
    ]

    missing_regions = [
        region
        for region in region_columns
        if region not in df_regions.columns
    ]

    if missing_regions:
        raise ValueError(
            "The following region columns are missing from the region Excel file: "
            f"{missing_regions}"
        )

    country_groups = {}

    for region in region_columns:
        country_groups[region] = (
            df_regions.loc[
                df_regions[region] == 1,
                "Node"
            ]
            .astype(str)
            .str.strip()
            .tolist()
        )

    return country_groups


country_groups = load_country_groups(full_region_file_path)


# ============================================================
# 8. READ COST TABLE
# ============================================================

def read_cost_table(file_path):
    """Read the cost sheet from a prepared results file."""
    df = pd.read_excel(
        file_path,
        sheet_name="cost"
    )

    df["Node"] = (
        df["Node"]
        .astype(str)
        .str.strip()
    )

    return df


base_df = read_cost_table(base_file)

if "Total Cost" not in base_df.columns:
    raise ValueError("'Total Cost' column not found in the cost sheet.")


# ============================================================
# 9. READ FIXED GAS DEMAND
# ============================================================

supply_df = pd.read_excel(
    full_input_file_path,
    sheet_name="Supply"
)

# Keep methane only
demand_df = supply_df[
    supply_df["Commodity"]
    .astype(str)
    .str.strip()
    == "Methane"
].copy()

# Standardise node names
demand_df["Node"] = (
    demand_df["Node"]
    .astype(str)
    .str.strip()
)

# Demand is negative in the input file; use positive demand as the weight.
demand_df["Demand"] = demand_df["Supply"].abs()

demand_df = demand_df[["Node", "Demand"]].copy()

# Check duplicate demand entries
duplicate_nodes = (
    demand_df.loc[
        demand_df["Node"].duplicated(),
        "Node"
    ]
    .unique()
)

if len(duplicate_nodes) > 0:
    raise ValueError(
        "Duplicate methane demand entries found for: "
        f"{duplicate_nodes.tolist()}"
    )


# ============================================================
# 10. ADD BASE-CASE DEMAND WEIGHTS
# ============================================================

def add_demand_weights(cost_df, demand_df):
    """Merge the fixed methane demand weights with a cost table."""
    df = cost_df.merge(
        demand_df,
        on="Node",
        how="left",
        validate="one_to_one"
    )

    missing_demand = (
        df.loc[
            df["Demand"].isna(),
            "Node"
        ]
        .unique()
        .tolist()
    )

    if missing_demand:
        raise ValueError(
            "No methane demand found for: "
            f"{missing_demand}"
        )

    return df


base_df = add_demand_weights(base_df, demand_df)


# ============================================================
# 11. READ ALL SENSITIVITY COST TABLES
# ============================================================

sensitivity_dfs = {}

for sensitivity_name, file_path in sensitivity_files.items():
    df = read_cost_table(file_path)

    if "Total Cost" not in df.columns:
        raise ValueError(
            f"'Total Cost' column not found in {sensitivity_name}."
        )

    # Demand is unchanged across sensitivities, so always use the base-case weights.
    df = add_demand_weights(df, demand_df)

    sensitivity_dfs[sensitivity_name] = df


# ============================================================
# 12. COUNTRY-LEVEL PERCENTAGE CHANGES
# ============================================================

def calculate_country_changes(base_df, sensitivity_dfs):
    """Calculate each country's Total Cost change relative to the base case."""
    result = base_df[["Node", "Total Cost"]].copy()
    result = result.rename(columns={"Total Cost": "Base"})

    for sensitivity_name, sensitivity_df in sensitivity_dfs.items():
        sensitivity_values = sensitivity_df[["Node", "Total Cost"]].copy()
        sensitivity_values = sensitivity_values.rename(
            columns={"Total Cost": sensitivity_name}
        )

        result = result.merge(
            sensitivity_values,
            on="Node",
            how="left",
            validate="one_to_one"
        )

        result[sensitivity_name] = (
            result[sensitivity_name] / result["Base"] - 1
        ) * 100

    return result


country_results = calculate_country_changes(
    base_df,
    sensitivity_dfs
)


# ============================================================
# 13. REGIONAL DEMAND-WEIGHTED TOTAL COST
# ============================================================

def calculate_regional_cost(df, countries):
    """
    Calculate demand-weighted average Total Cost for a region.

    Regional cost = sum(Total Cost_i * Demand_i) / sum(Demand_i)

    The demand weights are fixed from the base input file.
    """
    subset = df[df["Node"].isin(countries)].copy()

    if subset.empty:
        return float("nan")

    total_demand = subset["Demand"].sum()

    if total_demand == 0:
        return float("nan")

    weighted_cost = (
        subset["Total Cost"] * subset["Demand"]
    ).sum() / total_demand

    return weighted_cost


# ============================================================
# 14. REGIONAL PERCENTAGE CHANGES
# ============================================================

def calculate_regional_changes(
    base_df,
    sensitivity_dfs,
    country_groups
):
    """Calculate demand-weighted regional Total Cost changes."""
    rows = []

    for region, countries in country_groups.items():
        base_value = calculate_regional_cost(
            base_df,
            countries
        )

        row = {
            "Node": region,
            "Base": base_value
        }

        for sensitivity_name, sensitivity_df in sensitivity_dfs.items():
            sensitivity_value = calculate_regional_cost(
                sensitivity_df,
                countries
            )

            if pd.isna(base_value) or base_value == 0:
                change = float("nan")
            else:
                change = (
                    sensitivity_value / base_value - 1
                ) * 100

            row[sensitivity_name] = change

        rows.append(row)

    return pd.DataFrame(rows)


regional_results = calculate_regional_changes(
    base_df,
    sensitivity_dfs,
    country_groups
)


# ============================================================
# 15. EXPORT RESULTS TO EXCEL
# ============================================================

output_file_name = (
    f"regional_country_sensitivity_{base_scenario}.xlsx"
)

output_file_path = os.path.join(
    full_prepared_results_path,
    output_file_name
)

with pd.ExcelWriter(
    output_file_path,
    engine="openpyxl"
) as writer:
    country_results.to_excel(
        writer,
        sheet_name="Countries",
        index=False
    )

    regional_results.to_excel(
        writer,
        sheet_name="Regions",
        index=False
    )


# ============================================================
# 16. DISPLAY FINAL TABLES
# ============================================================

country_results

regional_results

print(f"Results exported to:\n{output_file_path}")
